# Solutions: Ship It Without Forgetting

**Language:** Python
**Topics:** Instance Amnesia, 3-Axis debugging, Restart Test, Replay Tax, trim_messages, LangGraph persistence, thread_id, checkpointers
**Level:** Intermediate

Every answer is worked and run. Production questions get a live demonstration: you will see memory get wiped, then made durable, on real code.

Run the scaffold once, then read top to bottom. Same offline `LocalAgent` as notebook 1, plus LangGraph and trimming.

## Scaffold

In [ ]:
# pip install langchain langchain-core langgraph langgraph-checkpoint-sqlite
import os, sqlite3
import re, warnings
warnings.filterwarnings("ignore", message=".*RunnableWithMessageHistory.*")
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

class LocalAgent(BaseChatModel):
    # Offline stand-in. Finds a PNR in the messages it is given and answers from it.
    # No API key. Its job is to make the plumbing visible, not to be clever.
    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        seen = " ".join(m.content for m in messages if isinstance(m.content, str))
        mm = re.search(r"PNR\s+([A-Z0-9]{5,8})", seen)
        pnr = mm.group(1) if mm else None
        last = next((m.content for m in reversed(messages)
                     if m.type == "human" and isinstance(m.content, str)), "")
        low = last.lower()
        if "pnr" in low and "?" in last:
            r = f"Your PNR is {pnr}." if pnr else "I do not have your PNR. Could you share it?"
        elif "cancel" in low or "leg" in low:
            r = "I see the BLR to DEL leg on your booking. I can help with that."
        elif last:
            r = f"Noted: {last}"
        else:
            r = "How can I help with your booking today?"
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=r))])
    @property
    def _llm_type(self):
        return "local-agent"

def show(title, messages):
    print(f"[{title}]  {len(messages)} message(s) the model sees:")
    for i, m in enumerate(messages, 1):
        text = m.content if isinstance(m.content, str) else str(m.content)
        print(f"   {i}. {m.type:6} | {text}")

model = LocalAgent()
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise airline support agent."),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])
chain = prompt | model | StrOutputParser()
print("Scaffold ready. Model:", type(model).__name__)

from langchain_core.messages import trim_messages
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.checkpoint.memory import MemorySaver

def make_rwmh_bot(store):
    def get_session_history(sid):
        if sid not in store:
            store[sid] = InMemoryChatMessageHistory()
        return store[sid]
    return RunnableWithMessageHistory(chain, get_session_history,
        input_messages_key="input", history_messages_key="history")

def bar(n):
    return "#" * n

print("LangGraph + trim ready.")

## Q1. Predict the output  ·  Predict

**Answer**
```
[2, 4, 6]
12
```

**Why, step by step**
1. Start with one system message.
2. Each turn appends the human line, so the count re-sent that turn is what is on the list at that moment: 2, then 4, then 6.
3. The running total (12) grows faster than the number of turns (3).

That gap is the tax. Full-history memory is quadratic in conversation length:

$$
Total \approx \sum_{n=1}^{N}\left(system + \sum_{i=1}^{n-1} msg_i + input_n\right) = O(N^2)
$$

In [ ]:
convo = [SystemMessage("Airline support.")]
per_turn = []
for t in ["one", "two", "three"]:
    convo.append(HumanMessage(t))
    per_turn.append(len(convo))
    convo.append(AIMessage("ack"))

print("per_turn:", per_turn)
print("sum     :", sum(per_turn))
print()
for i, n in enumerate(per_turn, 1):
    print(f"turn {i}: {bar(n)} ({n})")

## Q2. Predict the output  ·  Predict

**Answer**
```
11 4
['system', 'ai', 'human', 'ai']
```

**Why, step by step**
1. The conversation has 11 messages (1 system + 5 pairs).
2. `token_counter=len` counts one per message, so `max_tokens=4` keeps four.
3. `include_system=True` pins the system message in.
4. `strategy="last"` fills the rest with the three most recent: a3, q4, a4.

The recent window starts on an ai turn with no matching human. That runs, but it is blunt.

In [ ]:
convo = [SystemMessage("s")]
for i in range(5):
    convo.append(HumanMessage(f"q{i}"))
    convo.append(AIMessage(f"a{i}"))

trimmed = trim_messages(convo, max_tokens=4, strategy="last",
                        token_counter=len, include_system=True)
print(len(convo), len(trimmed))
print([m.type for m in trimmed])

**Skeptic asks:** a window that begins on a lone ai turn can confuse some models. Pass `start_on="human"` and the window starts cleanly on a human turn. Compare the two below.

In [ ]:
clean = trim_messages(convo, max_tokens=4, strategy="last",
                      token_counter=len, include_system=True, start_on="human")
print("default   :", [m.type for m in trimmed])
print("start_on  :", [m.type for m in clean])

## Q3. Read and classify  ·  Read

**Answer: (c) WHERE.** Each server keeps its own in-process store, so a request routed to a fresh replica sees no history. The id is right and the wiring is right; the storage is not shared.

```mermaid
flowchart TB
    WHO[WHO: identity is session_id] --> OK1[fine]
    HOW[HOW: wiring is RunnableWithMessageHistory] --> OK2[fine]
    WHERE[WHERE: storage is a per-process dict] --> BUG[the fault: not shared across servers]
```

Fix: one shared backend, Redis or Postgres. "The bot forgot" is almost always WHO or WHERE, rarely the model.

## Q4. Multi-select  ·  Read

**Answer: pass are** SQLite file, Redis, Postgres. **Fail:** in-memory dict and a module-global list.

| Storage | Survives restart | Safe across replicas |
|---|---|---|
| in-memory dict | no | no |
| module-global list | no | no |
| SQLite file | yes | no |
| Redis | yes | yes |
| Postgres | yes | yes |

The Restart Test in one line: if this process restarts right now, does the customer's history survive? Anything living in RAM answers no.

## Q5. True or false  ·  Read

| # | Statement | Verdict |
|---|---|---|
| 1 | `MemorySaver` survives a restart | False |
| 2 | `SqliteSaver` writes to a file, survives a restart | True |
| 3 | swapping the saver changes graph logic | False |
| 4 | `PostgresSaver` is the rung past a single instance | True |

Below is the proof. Same graph, two savers. `MemorySaver` forgets across a restart. `SqliteSaver` reads the same file and remembers.

In [ ]:
def call(state):
    return {"messages": [model.invoke(state["messages"])]}

def build(saver):
    b = StateGraph(MessagesState)
    b.add_node("agent", call)
    b.add_edge(START, "agent")
    return b.compile(checkpointer=saver)

cfg = {"configurable": {"thread_id": "rao"}}

# MemorySaver: a "restart" is a brand new saver with nothing in it.
g1 = build(MemorySaver())
g1.invoke({"messages": [HumanMessage("I am Rao, PNR JX48Q2.")]}, config=cfg)
g2 = build(MemorySaver())   # restart
print("MemorySaver after restart:",
      len(g2.get_state(cfg).values.get("messages", [])), "messages (gone)")

In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver

DB = "sol2_ckpt.sqlite"
if os.path.exists(DB):
    os.remove(DB)

conn = sqlite3.connect(DB, check_same_thread=False)
gs = build(SqliteSaver(conn))
gs.invoke({"messages": [HumanMessage("I am Rao, PNR JX48Q2.")]}, config=cfg)
conn.close()                                   # process exits

conn2 = sqlite3.connect(DB, check_same_thread=False)   # restart, same file
gs2 = build(SqliteSaver(conn2))
print("SqliteSaver after restart:",
      len(gs2.get_state(cfg).values["messages"]), "messages (survived)")
conn2.close()
os.remove(DB)

## Q6. Order the ladder  ·  Order

**Answer, lowest first:**

```mermaid
flowchart LR
    A[1 in-memory dict] --> B[2 SQLite file] --> C[3 Redis] --> D[4 Postgres] --> E[5 LangGraph checkpointer on Postgres]
```

Each rung buys back one failure: restart, then concurrency, then horizontal scale, then resumable agent state. Climb only as far as your traffic forces you to.

## Q7. Match symptom to root cause  ·  Match

| Symptom | Root cause |
|---|---|
| forgot after the overnight update | restart wiped in-process RAM |
| forgets randomly, only sometimes | requests hit different replicas with unshared storage |

```mermaid
flowchart TB
    subgraph Restart
      D1[dict has chats] --> RD[redeploy] --> E1[dict empty]
    end
    subgraph Autoscale
      REQ[request] --> LB[load balancer]
      LB --> RA[replica A has history]
      LB --> RB[replica B empty]
    end
```

"Overnight" points at a restart. "Randomly" points at replicas.

## Q8. Debug: Instance Amnesia  ·  Debug

**The buggy code**
```python
replica_A, replica_B = {}, {}
botA = make_rwmh_bot(replica_A)
botB = make_rwmh_bot(replica_B)   # a second, separate store
```

**Bad:** two dicts, so turn two on `botB` never sees turn one on `botA`. **Fix:** one shared store behind both bots.

Watch it forget, then fix it.

In [ ]:
# BROKEN: separate stores per replica
replica_A, replica_B = {}, {}
botA = make_rwmh_bot(replica_A)
botB = make_rwmh_bot(replica_B)

botA.invoke({"input": "I am Rao, PNR JX48Q2."},
            config={"configurable": {"session_id": "rao"}})
lost = botB.invoke({"input": "What is my PNR?"},
                   config={"configurable": {"session_id": "rao"}})
print("Split replicas ->", lost)

In [ ]:
# FIXED: one shared store (a stand-in for Redis or Postgres)
shared = {}
botA = make_rwmh_bot(shared)
botB = make_rwmh_bot(shared)

botA.invoke({"input": "I am Rao, PNR JX48Q2."},
            config={"configurable": {"session_id": "rao"}})
found = botB.invoke({"input": "What is my PNR?"},
                    config={"configurable": {"session_id": "rao"}})
print("Shared store ->", found)

## Q9. Debug: no checkpointer  ·  Debug

**The buggy code**
```python
graph = b.compile()          # no checkpointer
graph.get_state(cfg)         # raises
```

**Bad line:** `b.compile()` has no checkpointer, so there is no memory to save or read. **Fix:** `b.compile(checkpointer=MemorySaver())`.

The error is loud and specific. See it, then fix it.

In [ ]:
def call(state):
    return {"messages": [model.invoke(state["messages"])]}

b = StateGraph(MessagesState)
b.add_node("agent", call)
b.add_edge(START, "agent")

# BROKEN
graph = b.compile()
cfg = {"configurable": {"thread_id": "rao"}}
graph.invoke({"messages": [HumanMessage("I am Rao, PNR JX48Q2.")]}, config=cfg)
try:
    graph.get_state(cfg)
except Exception as e:
    print("Error you get:", type(e).__name__, "-", e)

In [ ]:
# FIXED
graph = b.compile(checkpointer=MemorySaver())
graph.invoke({"messages": [HumanMessage("I am Rao, PNR JX48Q2.")]}, config=cfg)
print("state now holds:", len(graph.get_state(cfg).values["messages"]), "messages")

## Q10. Debug: trimming drops the system message  ·  Debug

**The buggy code**
```python
recent = trim_messages(convo, max_tokens=4, strategy="last", token_counter=len)
# include_system defaults to False -> system line is trimmed away
```

**Bad:** `include_system` defaults to False, so the agent loses its instructions. **Fix:** add `include_system=True`.

Watch the system message vanish, then stay.

In [ ]:
convo = [SystemMessage("You are a concise airline support agent.")]
for i in range(5):
    convo.append(HumanMessage(f"q{i}"))
    convo.append(AIMessage(f"a{i}"))

# BROKEN
recent = trim_messages(convo, max_tokens=4, strategy="last", token_counter=len)
print("BROKEN types:", [m.type for m in recent], " <- no system")

In [ ]:
# FIXED
recent = trim_messages(convo, max_tokens=4, strategy="last",
                       token_counter=len, include_system=True)
print("FIXED types :", [m.type for m in recent], " <- system kept")

## Q11. Diagram to code: durable agent  ·  Build

```mermaid
flowchart LR
    REQ[request with thread id] --> G[LangGraph agent node]
    G --> M[model call]
    G --> CK[(SqliteSaver on disk)]
    CK --> FILE[(file survives restart)]
```

**Solution steps**
1. Wrap a sqlite connection in `SqliteSaver`.
2. `compile(checkpointer=saver)`.
3. Invoke with a `thread_id`.

Then the payoff: reopen the same file and the history is still there.

In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver

def call_model(state):
    return {"messages": [model.invoke(state["messages"])]}

def build_durable_graph(conn):
    saver = SqliteSaver(conn)
    b = StateGraph(MessagesState)
    b.add_node("agent", call_model)
    b.add_edge(START, "agent")
    return b.compile(checkpointer=saver)

DB = "sol2_durable.sqlite"
if os.path.exists(DB):
    os.remove(DB)

conn = sqlite3.connect(DB, check_same_thread=False)
graph = build_durable_graph(conn)
cfg = {"configurable": {"thread_id": "cust-rao"}}
graph.invoke({"messages": [HumanMessage("I am Rao, PNR JX48Q2.")]}, config=cfg)
graph.invoke({"messages": [HumanMessage("What is my PNR?")]}, config=cfg)
print("wrote", len(graph.get_state(cfg).values["messages"]), "messages, then closing")
conn.close()

In [ ]:
# Restart: fresh connection and graph from the SAME file
conn2 = sqlite3.connect(DB, check_same_thread=False)
graph2 = build_durable_graph(conn2)
survived = graph2.get_state(cfg).values["messages"]
print("after restart:", len(survived), "messages survived")
print("answer still correct ->",
      graph2.invoke({"messages": [HumanMessage("Remind me, what is my PNR?")]},
                    config=cfg)["messages"][-1].content)
conn2.close()
os.remove(DB)

## Q12. Diagram to code: a capped node  ·  Build

```mermaid
flowchart LR
    STATE[state messages] --> TRIM[keep system plus last 6] --> CALL[model invoke] --> OUT[new ai message]
```

**Solution:** trim inside the node, then invoke. The model never sees more than seven messages, no matter how long the thread grows. This is where the Replay Tax gets paid down.

In [ ]:
def call_model_cap(state):
    recent = trim_messages(state["messages"], max_tokens=6, strategy="last",
                           token_counter=len, include_system=True)
    return {"messages": [model.invoke(recent)]}

# prove the cap: feed a long thread, check what the node would send
long_state = {"messages": [SystemMessage("Airline support.")]}
for i in range(10):
    long_state["messages"].append(HumanMessage(f"msg {i}"))
    long_state["messages"].append(AIMessage(f"ack {i}"))

capped = trim_messages(long_state["messages"], max_tokens=6, strategy="last",
                       token_counter=len, include_system=True)
print("thread length:", len(long_state["messages"]))
print("model sees   :", len(capped), "messages ->", [m.type for m in capped])

## Q13. Case study  ·  Predict

**Answers**
1. Replica B replies `I do not have your PNR. Could you share it?`
2. The failure is Instance Amnesia.
3. The fix is one shared, durable backend (Redis or Postgres).

Replica B's store never saw turn one, so it has nothing to read. Demonstrated below with an empty history.

In [ ]:
# Replica B handles turn two with an empty history for Rao
print("Replica B reply ->", model.invoke([HumanMessage("What is my PNR?")]).content)
print("Diagnosis: Instance Amnesia. Fix: shared store, not per-process memory.")

## Q14. Predict the output  ·  Predict

**Answer**
```
4
2
```

**Why, step by step**
1. Each `invoke` on a thread adds a human and an ai message.
2. `rao` ran twice, so 4. `mehta` ran once, so 2.
3. Different `thread_id` means a different checkpoint. The raw graph has no prompt template, so no system message is added and counts are pure turn pairs.

In [ ]:
def call(state):
    return {"messages": [model.invoke(state["messages"])]}

b = StateGraph(MessagesState)
b.add_node("agent", call)
b.add_edge(START, "agent")
graph = b.compile(checkpointer=MemorySaver())

rao = {"configurable": {"thread_id": "rao"}}
mehta = {"configurable": {"thread_id": "mehta"}}

graph.invoke({"messages": [HumanMessage("I am Rao, PNR JX48Q2.")]}, config=rao)
graph.invoke({"messages": [HumanMessage("What is my PNR?")]}, config=rao)
graph.invoke({"messages": [HumanMessage("I am Mehta, PNR ZZ90Q1.")]}, config=mehta)

print(len(graph.get_state(rao).values["messages"]))
print(len(graph.get_state(mehta).values["messages"]))

**Skeptic asks:** the counts differ, but did the threads truly stay separate? Test it directly below. Ask for Rao's PNR on the `mehta` thread and you get Mehta's, not Rao's. Isolation is keyed entirely by `thread_id`.

In [ ]:
cross = graph.invoke({"messages": [HumanMessage("What is my PNR?")]}, config=mehta)
print("Ask on mehta thread ->", cross["messages"][-1].content)

You reproduced the two production regressions (missing checkpointer, missing `include_system`), watched memory get wiped and made durable, and proved thread isolation. That is the shippable version of the notepad.